# The `fm` CLI — pretrain → finetune → inverse → predict

The whole foundation-model workflow runs through a single console command, **`fm`**, a
[click](https://click.palletsprojects.com/) group with four subcommands:

| Subcommand | What it does | Engine |
|---|---|---|
| `fm pretrain` | continual-rehearsal pre-training (rehearsal-interval replay + optional `n_runs` sweep) | `workflows/pretrain.py` |
| `fm finetune` | frozen-encoder fine-tuning of selected heads (AE head stays trainable) | `workflows/finetune.py` |
| `fm inverse`  | inverse design (scenarios × latent/composition paths) | `workflows/inverse.py` |
| `fm predict`  | evaluate / predict with an arbitrary checkpoint | `workflows/predict.py` |

Every subcommand reads a **TOML** config, writes `run_provenance.json` + `run.log` into its
output dir, and shares the `[data]`/`[descriptor]`/`[datasets.*]`/`[[tasks]]`/`[model]`/`[training]`
sections plus one subcommand section. This notebook walks the smoke chain (`samples/*_smoke.toml`,
CPU-friendly) both from the shell and programmatically.

## 1. The config

The smoke configs run the full chain on the qc dataset in minutes. Here is the pretrain one:

In [ ]:
from pathlib import Path

print(Path("../samples/pretrain_smoke.toml").read_text())

## 2. From the shell

Each subcommand is a one-liner. Common flags on all of them: `--config` (required), `--output-dir`,
`--set section.key=value` (repeatable, TOML-valued), `--seed`, `--accelerator`, `--sample`.

```bash
# 1. pre-train (2 reg + 1 clf head over the qc dataset, 2-run sweep)
fm pretrain --config samples/pretrain_smoke.toml

# 2. fine-tune one head on the pretrain checkpoint
fm finetune --config samples/finetune_smoke.toml \
    --checkpoint artifacts/pretrain_smoke/runs/run00/training/final_model.pt

# 3. inverse design (1 scenario x 2 paths)
fm inverse  --config samples/inverse_smoke.toml \
    --checkpoint artifacts/finetune_smoke/training/final_model.pt

# 4. evaluate on the test split
fm predict  --config samples/predict_smoke.toml \
    --checkpoint artifacts/finetune_smoke/training/final_model.pt
```

Run `fm --help` or `fm <subcommand> --help` for the full flag list.

## 3. Programmatically

The CLI is a thin wrapper — each subcommand is `build_*_config(raw_toml) -> run(cfg, recorder)`.
That is handy for embedding a run in a script or notebook. Below we drive the whole chain on the
smoke configs (CPU, a couple of minutes). Uncomment the last cell to execute.

In [ ]:
import tomllib
from pathlib import Path

from foundation_model.workflows.pretrain import build_pretrain_config, run as pretrain_run
from foundation_model.workflows.finetune import build_finetune_config, run as finetune_run
from foundation_model.workflows.inverse import build_inverse_config, run as inverse_run
from foundation_model.workflows.predict import build_predict_config, run as predict_run
from foundation_model.workflows.recording import RunRecorder

SAMPLES = Path("../samples")


def run_stage(build, run_fn, toml_name, *, checkpoint=None):
    raw = tomllib.load(open(SAMPLES / toml_name, "rb"))
    cfg = build(raw, checkpoint=checkpoint) if checkpoint is not None else build(raw)
    rec = RunRecorder(cfg.output_dir)
    seed = cfg.training.seed if hasattr(cfg, "training") else getattr(cfg, "seed", 2025)
    rec.write_provenance(config=cfg, argv=["fm", toml_name], seeds={"seed": seed})
    result = run_fn(cfg, rec)
    rec.close()
    return cfg, result

In [ ]:
# --- uncomment to run the full smoke chain (needs data/qc_*.pd.parquet) ---
# pre_cfg, _ = run_stage(build_pretrain_config, pretrain_run, "pretrain_smoke.toml")
# pre_ckpt = pre_cfg.output_dir / "runs" / "run00" / "training" / "final_model.pt"
#
# ft_cfg, _ = run_stage(build_finetune_config, finetune_run, "finetune_smoke.toml", checkpoint=pre_ckpt)
# ft_ckpt = ft_cfg.output_dir / "training" / "final_model.pt"
#
# run_stage(build_inverse_config, inverse_run, "inverse_smoke.toml", checkpoint=ft_ckpt)
# run_stage(build_predict_config, predict_run, "predict_smoke.toml", checkpoint=ft_ckpt)

## 4. Inspecting the outputs

Every run drops a provenance file and a log; each engine adds its own artifacts (per-step
checkpoints + parquet predictions + metrics for pretrain/finetune/predict, and per-scenario
figures + trajectory `.npz` for inverse).

In [ ]:
import json
import pandas as pd

run_dir = Path("../artifacts/predict_smoke")  # point at a run you produced in section 3
prov = run_dir / "run_provenance.json"
if prov.exists():
    p = json.loads(prov.read_text())
    print("packages:", p["packages"])
    print("git:", p["git"], "| seeds:", p["seeds"])
    metrics = run_dir / "predict" / "metrics.json"
    if metrics.exists():
        print("metrics:", json.loads(metrics.read_text()))
    preds = run_dir / "predict" / "density_pred.parquet"
    if preds.exists():
        display(pd.read_parquet(preds).head())
else:
    print("Run a chain in section 3 first (or point run_dir at an existing artifacts/ run).")